### Functions: `custom_logs` & `finalize_logs`

The `custom_logs` functions sets up a custom logging system for Databricks notebooks, allowing for logging of messages both to the console and a log file, with added metadata for better traceability. This function is reusable across multiple notebooks and organizes logs by notebook name and timestamp.

The `finalize_logs` function is used to flush the log buffer, close the file handler, and copy the log file from Databricks FileStore to Azure Blob Storage. It also removes the temporary log file from the FileStore once it has been successfully copied to the Blob Storage.

#### Parameters:
- **`notebook_name` (str)**: The name of the notebook. It is used to organize logs by notebook and create log file names. The default value is `"default_notebook"`.
- **`logger` (CustomAdapter)**: The custom logger instance created by the custom_logs function.
- **`storage_account_name` (str)**: The name of the Azure Storage account name Will be fetched directly from `"Workspace/Shared/Storage_Config"`.
- **`abfss_path` (str)**: The path in the Azure Blob Storage container where the log file will be copied.

#### Key Features:
1. **Log File Naming**:
   - The log file is named based on the notebook name and the current timestamp (`YYYYMMDD_HHMMSS`) to ensure uniqueness.
   - The log file is saved in `/dbfs/FileStore/logs/{notebook_name}/`.

2. **Log Handlers**:
   - **File Handler**: Logs messages with level `DEBUG` or higher to a file.
   - **Console Handler**: Logs messages with level `INFO` or higher to the console for real-time visibility.

3. **Custom Metadata**:
   - Adds the following metadata to each log message:
     - **Username**: The username of the individual running the notebook (`getpass.getuser()`).
     - **Calling Function**: The name of the function or method that generated the log message.

4. **Custom Log Format**:
   - Logs include:
     - Timestamp
     - Log level (`INFO`, `DEBUG`, `ERROR`, etc.)
     - Username
     - Calling function
     - Log message

5. **Thread-Safe Adapter**:
   - Uses `CustomAdapter` to dynamically include the username and calling function in each log entry.

6. **Reusable Across Notebooks**:
   - The function can be called from any notebook to set up a unique logger tailored to that notebook's context.

#### Returns:
- **`CustomAdapter`**: A customized logger instance that supports additional metadata in log messages.

#### Returns:
- **`finalize_logs`**: None

#### Example Usage:
```python
logger = custom_logs(notebook_name="data_processing_notebook")
logger.info("This is an informational message.")
logger.debug("Debugging details.")
logger.error("An error occurred.")
finalize_logs(logger, storage_account_name, abfss_path)



In [0]:
%run /Workspace/Shared/Storage_Config

In [0]:
import logging
import os
import getpass
import inspect
from datetime import datetime

def custom_logs(notebook_name=""):
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.DEBUG)

    current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"/dbfs/FileStore/logs/{notebook_name}"
    log_file = os.path.join(log_dir, f"{notebook_name}_log_{current_time}.log")
    os.makedirs(os.path.dirname(log_file), exist_ok=True)

    file_handler = logging.FileHandler(log_file, mode='a')
    console_handler = logging.StreamHandler()

    file_handler.setLevel(logging.DEBUG)
    console_handler.setLevel(logging.INFO)

    log_format = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(username)s - %(calling_func)s - %(message)s'
    )

    file_handler.setFormatter(log_format)
    console_handler.setFormatter(log_format)

    if logger.hasHandlers():
        logger.handlers.clear()

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    class CustomAdapter(logging.LoggerAdapter):
        def process(self, msg, kwargs):
            user = getpass.getuser()
            func_name = inspect.stack()[2][3]
            kwargs["extra"] = {'username': user, 'calling_func': func_name}
            return msg, kwargs

    logger.log_file = log_file 
    logger.file_handler = file_handler
    return CustomAdapter(logger, {})

In [0]:
def finalize_logs(logger, storage_account_name, abfss_path):
    log_file = logger.logger.log_file.split("/dbfs")[-1]
    blob_path = f"abfss://computation@{storage_account_name}.dfs.core.windows.net/{abfss_path}"

    try:
        logger.logger.file_handler.flush()
        logger.logger.file_handler.close()

        dbutils.fs.cp(f"dbfs:{log_file}", blob_path)
        print(f"Log file copied to BLOB: {blob_path}")

        dbutils.fs.rm(f"dbfs:{log_file}")
        print(f"Temporary log file removed: {log_file}")
    except Exception as e:
        print(f"Failed to copy or remove log file: {e}")

In [0]:
notebook_name = "test_notebook"
folder_path = "100018_00_1/1/LOGS"
abfss_path = f"{folder_path}/{notebook_name}_log_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.log"

logger = custom_logs(notebook_name)

logger.info('This is an informational message.')
logger.debug('This is a debug message.')
logger.error('This is an error message.')

finalize_logs(logger, storage_account_name, abfss_path)